# RAG evaluation with LLM-as-a-judge

End-to-end evaluation of the Wikifin assistant:

1. Sample Dutch document chunks from the index
2. Generate questions with an LLM — in Dutch, English and French
3. Run `get_assistant_response` for each question
4. Score each answer with a separate judge LLM on relevance
5. Observe scores overall and by query language

In [1]:
from __future__ import annotations

import random
from collections import defaultdict
from tqdm.auto import tqdm


from wikifin_rag.assistant import get_assistant_response
from wikifin_rag.evaluation_utils import sample_documents, generate_corpus_queries
from wikifin_rag.config import PROJECT_ROOT
from wikifin_rag.judge import evaluate_anwer

import pandas as pd


# Evaluation knobs
SAMPLE_SIZE = 40
SEED = 42
DOC_LANGUAGE = "nl"
# Fraction of examples whose question is generated in a non-Dutch language
NON_DUTCH_FRACTION = 0.4
QUERY_LANGUAGES_NON_DUTCH = ["en", "fr"]
INCLUDE_FIELDS = ["title", "section", "text", "source_url", "language"]

QUERY_LANGUAGES = ["nl", "en", "fr"]

In [2]:
_DATA_PATH = PROJECT_ROOT / "data"
_DATA_PATH.mkdir(exist_ok=True, parents=True)

## 1. Sample documents

In [3]:
random.seed(SEED)

## 2. Build evaluation cases

Each case has a question and query language.

In [4]:
try:
    df_ground_truth = pd.read_csv(_DATA_PATH / "ground_truth-new.csv")
    ground_truth = df_ground_truth.to_dict(orient="records")
except FileNotFoundError:
    docs = sample_documents(include_fields=INCLUDE_FIELDS)
    ground_truth, total_cost = generate_corpus_queries(docs, file_path=_DATA_PATH / "ground_truth-new.csv")
    df_ground_truth = pd.DataFrame(ground_truth)
    print(total_cost)

  0%|          | 0/20 [00:00<?, ?it/s]

0.01506


In [5]:
df_ground_truth.head()

,query,query_language,relevant_id,title,section,text
0,Hoe wordt ontvangen alimentatie belast bij de ...,nl,050dbbd0898802d4#0,De ontvangen alimentatie,De ontvangen alimentatie,De ontvangen alimentatie is een inkomen dat bi...
1,How is received alimony taxed for personal inc...,en,050dbbd0898802d4#0,De ontvangen alimentatie,De ontvangen alimentatie,De ontvangen alimentatie is een inkomen dat bi...
2,Comment la pension alimentaire reçue est-elle ...,fr,050dbbd0898802d4#0,De ontvangen alimentatie,De ontvangen alimentatie,De ontvangen alimentatie is een inkomen dat bi...
3,Wat kan ik doen als mijn verzekeraar door mijn...,nl,079bb6a60242280a#0,Wat is de schuldsaldoverzekering?,Als je de premie van je schuldsaldoverzekering...,Hoe hoger de verzekeraar het risico inschat da...
4,What can I do if my insurer asks for a higher ...,en,079bb6a60242280a#0,Wat is de schuldsaldoverzekering?,Als je de premie van je schuldsaldoverzekering...,Hoe hoger de verzekeraar het risico inschat da...


In [6]:
sample_docs = df_ground_truth.sample(n=SAMPLE_SIZE, random_state=SEED).to_dict(orient="records")
print(f"Sampled {len(sample_docs)} documents")
sample_docs[0]

Sampled 40 documents


{'query': 'Hoe wordt ontvangen alimentatie belast bij de personenbelasting?',
 'query_language': 'nl',
 'relevant_id': '050dbbd0898802d4#0',
 'title': 'De ontvangen alimentatie',
 'section': 'De ontvangen alimentatie',
 'text': 'De ontvangen alimentatie is een inkomen dat bij de andere inkomsten wordt gevoegd en wordt belast in de hoogste belastingschijven.'}

In [ ]:
def get_llm_response(question: str):
    try:
        exchange_record, *_ = get_assistant_response(question=question)
        llm_answer = exchange_record.response
        error = None
    except Exception as exc:
        llm_answer = None
        error = str(exc)
    
    return llm_answer, error

In [8]:
def generate_rag_answers(cases):
    eval_cases: list[dict] = []
    for case in tqdm(cases):
        if case:
            answer, error = get_llm_response(case["query"])

            eval_cases.append({
                "doc_id": case["relevant_id"],
                "query_language": case["query_language"],
                "query": case["query"],
                "answer": answer,
                "error": error,
                "reference_context": {
                    "title": case["title"],
                    "section": case["section"],
                    "text": (case.get("text") or "")[:1500]
                }
            })

    return eval_cases

In [9]:
try:
    df_cases = pd.read_csv(_DATA_PATH / "rag-answers.csv")
    eval_cases = df_cases.to_dict(orient="records")
except FileNotFoundError:
    eval_cases = generate_rag_answers(sample_docs)
    df_cases = pd.DataFrame(eval_cases)
    df_cases.to_csv(_DATA_PATH / "rag-answers.csv", index=False)

  0%|          | 0/40 [00:00<?, ?it/s]

In [10]:
lang_counts = defaultdict(int)
for case in eval_cases:
    lang_counts[case["query_language"]] += 1
print(f"Built {len(eval_cases)} cases: {dict(lang_counts)}")
eval_cases[0]

Built 40 cases: {'nl': 15, 'fr': 11, 'en': 14}


{'doc_id': '050dbbd0898802d4#0',
 'query_language': 'nl',
 'query': 'Hoe wordt ontvangen alimentatie belast bij de personenbelasting?',
 'answer': 'Ontvangen alimentatie voor jezelf moet je aangeven als **diverse inkomsten**. Ze wordt belast in de **personenbelasting aan de progressieve tarieven**, maar **slechts op 60% van het ontvangen bedrag**. Bijvoorbeeld: bij 2.400 euro ontvangen alimentatie wordt 1.440 euro belast. Als je in de hoogste schijf zit, kan dat neerkomen op 50% belasting op dat belastbare deel. [Wikifin: De ontvangen alimentatie](https://www.wikifin.be/nl/belasting-werk-en-inkomen/belastingaangifte/diverse-inkomsten-en-alimentatie/de-ontvangen)\n\nAls het gaat om **alimentatie voor een kind**, dan hoort die in principe bij het inkomen van het kind en niet bij dat van jou. [Wikifin: De ontvangen alimentatie](https://www.wikifin.be/nl/belasting-werk-en-inkomen/belastingaangifte/diverse-inkomsten-en-alimentatie/de-ontvangen)',
 'error': None,
 'reference_context': {'titl

## 3. LLM-as-a-judge

Each answer is scored on a simple discrete relevance scale.

In [ ]:
def get_llm_judgments(eval_cases) -> list[dict]:
    judged: list[dict] = []
    for case in tqdm(eval_cases):
        verdict = evaluate_anwer(query=case["query"], response=case["answer"])

        if not verdict:
            continue

        relevance, reasoning = verdict

        judged.append({
            **case,
            "relevance": relevance,
            "reasoning": reasoning,
        })

    return judged

In [12]:
try:
    df_judgments = pd.read_csv(_DATA_PATH / "llm-judgments.csv")
    judgments = df_judgments.to_dict(orient="records")
except FileNotFoundError:
    judgments = get_llm_judgments(eval_cases)
    df_judgments = pd.DataFrame(judgments)
    df_judgments.to_csv(_DATA_PATH / "llm-judgments.csv", index=False)

  0%|          | 0/40 [00:00<?, ?it/s]

In [13]:
print(f"Judged {len(judgments)}/{len(eval_cases)} cases")

Judged 40/40 cases


## 4. Aggregate results

In [14]:
display(df_judgments.groupby("relevance").size())

relevance
PARTLY_RELEVANT     1
RELEVANT           39
dtype: int64

### Bad answers

Inspect cases the judge rated poorly for debugging.

In [16]:
debug_cases = df_judgments.loc[
    df_judgments["relevance"] != "RELEVANT",
    ["doc_id", "query", "answer", "relevance", "reasoning"]
]

for case in debug_cases.itertuples():
    print(f"""
    ID: {case.doc_id}
    QUERY: {case.query}
    RESPONSE: {case.answer}
    RELEVANCE: {case.relevance}
    REASONING:
    {case.reasoning}
    """)


    ID: 06ff43808413a96c#0
    QUERY: À partir de quel âge peut-on ouvrir un compte à vue pour son enfant ?
    RESPONSE: Vanaf ongeveer 12 jaar is het openen van een zichtrekening interessant voor een kind. Wikifin vermeldt ook dat een zichtrekening al voor een jong kind een goed idee kan zijn, maar koppelt het moment waarop het echt interessant wordt aan die leeftijd. [Wikifin: “Zakgeld geven of niet?”](https://www.wikifin.be/nl/familie/financiele-opvoeding/zakgeld-geven-niet)
    RELEVANCE: PARTLY_RELEVANT
    REASONING:
    The assistant answers in Dutch and gives an age indication (around 12 years), which is related to the question. However, it does not directly answer the French question clearly, and the phrasing suggests the opening of a checking account is only 'interesting' from that age rather than stating from what age it can be opened. The response is therefore only partially aligned with the question.
    
